# L6a: Data-Driven Minimum-Variance Portfolios
In L5b, we estimated how asset growth rates vary together and sampled possible portfolio allocations. Today, we ask how to choose the weights: given the estimated mean growth rates and covariance matrix, how much of our wealth should we put in each asset? We use Markowitz's mean-variance framework to find allocations that minimize risk for a required reward.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate portfolio reward and risk:__ Derive the portfolio's expected growth rate and variance by treating its growth rate as a weighted average of the asset growth rates. Express the results using the weight vector, mean growth-rate vector, and covariance matrix, and explain how covariance affects risk and diversification.
> * __Construct minimum-variance portfolios:__ Use a Lagrange multiplier to derive the global minimum-variance weights when short positions are allowed. Formulate the target-growth problem, distinguish the minimum-variance frontier from its efficient branch, and explain how a long-only constraint and an equality or inequality growth target affect the feasible portfolios and the solution.
> * __Evaluate estimated allocations:__ Explain how the estimated means and covariance affect the selected weights, and compare portfolio performance on prices reserved for testing.

In this lecture, we begin with risky assets and use the estimated means and covariance to construct minimum-variance portfolios. We trace the efficient frontier and evaluate the selected allocations on a separate year of prices. We also consider how uncertainty in the estimated growth rates and covariances affects the chosen weights.

Let's get started!

___


## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Compute minimum-variance portfolios and the efficient frontier from data](CHEME-5660-L6a-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). How do historical growth rates and covariances determine our portfolio weights? We estimate the inputs from 2014 to 2024 data, compare allocations with and without short positions, and construct the efficient frontier. We compare the optimized portfolios with equal weights and an index fund using 2025 prices.

> [▶ Simulate a portfolio with multiple asset GBM](CHEME-5660-L6a-Example-MAGBM-Portfolio-Fall-2026.ipynb). How much might the wealth of a chosen portfolio vary? We use the multiple asset model of L5b to simulate correlated price paths and calculate buy-and-hold wealth. We compare the simulated wealth range with the observed 2025 path and an index fund, then estimate the probability of exceeding a target scaled net present value.

Optional examples on frontier geometry and estimation risk are listed in the Optional Advanced Material section at the end of this lecture.

___


## Concept Review: MAGBM, the Covariance Matrix, and Portfolio Wealth
In [L5b](../../week-5/L5b/CHEME-5660-L5b-Lecture-MultipleAsset-GBM-Fall-2026.ipynb), we used multiple asset geometric Brownian motion to describe correlated share prices. With constant parameters, the price of asset $i$ advances over a time step $\Delta t>0$ years according to:
$$
S_i(t+\Delta t)
=S_i(t)\exp\!\left[\mu_{g,i}\Delta t
+\sqrt{\Delta t}\,(\mathbf{A}\mathbf{Z})_i\right],
\qquad i=1,\ldots,M.
$$
Here, $\mu_{g,i}$ is the asset's mean growth rate, and the loading matrix satisfies $\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}$, where $\mathbf{C}$ is the covariance rate. The same vector $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}_M)$ supplies the random draws for all $M$ assets within a step; we draw a new independent vector at each step. The mean-growth vector is $\boldsymbol{\mu}_g$, and the growth-rate covariance is $\mathbf{\Sigma}_g=\mathbf{C}/\Delta t$.

From $N\geq2$ aligned growth-rate observations, the sample-mean vector $\mathbf{g}^{\prime}$ estimates $\boldsymbol{\mu}_g$. Subtracting each asset's sample mean gives the centered data matrix $\tilde{\mathbf{G}}\in\mathbb{R}^{N\times M}$. The covariance estimates are given by:
$$
\hat{\mathbf{\Sigma}}_g
=\frac{1}{N-1}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}},
\qquad
\hat{\mathbf{C}}=\Delta t\,\hat{\mathbf{\Sigma}}_g.
$$
We use $\hat{\mathbf{\Sigma}}_g$ to calculate portfolio growth-rate variance and $\hat{\mathbf{C}}$ to simulate prices. The sample covariance is positive semidefinite, so portfolio variances are non-negative. The closed-form weights developed today require positive definiteness, which the minimum-variance example checks.

If we did not finish the covariance example in L5b, we pick up its analysis now:

> __Example__
>
> [▶ Compute the covariance matrix for our dataset](../../week-5/L5b/CHEME-5660-L5b-Example-CovarianceMatrix-Fall-2026.ipynb). We estimate the growth-rate covariance, convert it to the GBM covariance rate, check the corresponding volatilities against L4b, and interpret the covariance and correlation of a pair of firms.

Recall also that initial weights $w_i\geq0$, with $\sum_iw_i=1$, divide initial wealth $W_0>0$ among the assets. Buying $n_i=w_iW_0/S_i(0)$ shares and holding them fixed gives the wealth and scaled NPV:
$$
\begin{aligned}
W_t&=W_0\sum_{i=1}^{M}w_i\frac{S_i(t)}{S_i(0)},\\
\rho_T&=\frac{W_T}{W_0}e^{-g_yT}-1.
\end{aligned}
$$
Here, $g_y$ is the constant benchmark growth rate and $T$ is the holding period in years. As in L5b, we allow fractional shares and omit dividends and trading costs. Simulated wealth outcomes let us estimate the probability of exceeding a target scaled NPV.

The Dirichlet distribution in L5b generated candidate allocations. Today, we choose weights by minimizing variance for a required expected growth rate.

___


## Modern Portfolio Theory: The Reward and Risk of a Portfolio
Modern Portfolio Theory (MPT) is a framework for choosing portfolio weights that maximizes expected growth rate for a given level of risk or, equivalently, minimizes risk for a given level of expected growth rate.

> __Reference:__ Modern Portfolio Theory was introduced by Harry Markowitz in the 1950s and became a standard framework in finance. Markowitz was awarded the Nobel Prize in Economic Sciences in 1990 for this work. The original publication is [Portfolio Selection, The Journal of Finance, Vol. 7, No. 1 (Mar., 1952), pp. 77-91](https://www.jstor.org/stable/2975974); the Nobel Prize information is [here](https://www.nobelprize.org/prizes/economic-sciences/1990/markowitz/facts/).

Let's warm up to MPT by watching a short video that introduces the key concepts: [here](https://www.youtube.com/watch?v=VsMpw-qnPZY). Markowitz's central insight was that an asset should not be judged in isolation: a volatile asset can reduce a portfolio's variance when its growth rate moves against the other holdings, and individually stable assets provide little diversification when they move together. To make that precise we need a reward and a risk for the portfolio as a whole, both written in terms of the weights.

### Portfolio reward
Consider a portfolio $\mathcal{P}=\{1,2,\ldots,M\}$ of $M$ risky assets with weight vector $\mathbf{w}=[w_1,\ldots,w_M]^{\top}$. Each $w_i$ is the fraction of the initial investment assigned to asset $i$, and the weights satisfy $\sum_iw_i=1$. We hold these weights fixed when calculating the portfolio's reward and risk.

Let $\mathbf{g}=[g_1,\ldots,g_M]^{\top}$ contain the asset growth rates over one observation interval. In the mean-variance model, we use their weighted average to describe portfolio growth:
$$
g_p=\sum_{i\in\mathcal{P}}w_i g_i=\mathbf{w}^{\top}\mathbf{g}.
$$
The asset growth rates are random, so $g_p$ is random as well. We measure __reward__ by its expected value. Because the weights are fixed, we can take them outside the expectation and use $\mathbb{E}[g_i]=\mu_{g,i}$ from L5b:
$$
\boxed{
\begin{aligned}
\mathbb{E}[g_p]
&=\mathbb{E}\!\left[\sum_{i\in\mathcal{P}}w_i g_i\right]\\
&=\sum_{i\in\mathcal{P}}w_i\mathbb{E}[g_i]
=\sum_{i\in\mathcal{P}}w_i\mu_{g,i}
=\mathbf{w}^{\top}\boldsymbol{\mu}_g.
\end{aligned}
}
$$
Thus, the portfolio's expected growth rate is the weighted average of the assets' mean growth rates. With time measured in years, reward has units of inverse years. In a calculation from data, the sample-mean vector $\mathbf{g}^{\prime}$ estimates $\boldsymbol{\mu}_g$, giving an estimated portfolio reward of $\mathbf{w}^{\top}\mathbf{g}^{\prime}$.

For a buy-and-hold investment, we calculate wealth from the fixed share counts and the asset prices. Over a period of length $\Delta t$, its log growth rate, when wealth remains positive, is given by:
$$
\frac{1}{\Delta t}\ln\!\left(\frac{W_{\Delta t}}{W_0}\right)
=\frac{1}{\Delta t}\ln\!\left(\sum_{i\in\mathcal{P}}w_i e^{g_i\Delta t}\right).
$$
The logarithm acts on the sum, so this generally differs from $\sum_iw_i g_i$. We use weighted growth rates to select the portfolio weights and the wealth formula to follow the resulting investment through time, as in L5b.

Expected growth describes the reward. Next, we need to account for how the asset growth rates vary together when calculating portfolio risk.

### Portfolio risk
We measure __risk__ by the variance of the weighted portfolio growth rate $g_p$. A larger variance means a wider spread of possible growth rates. To calculate it, we use the asset covariance matrix $\mathbf{\Sigma}_g=\operatorname{Cov}(\mathbf{g})$, whose entries are $\Sigma_{g,ij}=\operatorname{Cov}(g_i,g_j)$. Holding the weights fixed, the portfolio variance is given by:
$$
\boxed{
\begin{aligned}
\operatorname{Var}(g_p)
&=\operatorname{Cov}\!\left(\sum_{i\in\mathcal{P}}w_i g_i,\sum_{j\in\mathcal{P}}w_j g_j\right)\\
&=\sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_iw_j\operatorname{Cov}(g_i,g_j)
=\mathbf{w}^{\top}\mathbf{\Sigma}_g\mathbf{w}.
\end{aligned}
}
$$
The diagonal entries of the covariance matrix describe each asset's own variability. The off-diagonal entries describe how the asset growth rates vary together. Both contribute to portfolio risk. When working from data, we estimate $\mathbf{\Sigma}_g$ with the sample covariance $\hat{\mathbf{\Sigma}}_g$. Taking the square root of the portfolio variance gives the __growth-rate standard deviation__ $\sigma_{g,p}=\sqrt{\operatorname{Var}(g_p)}$, which has the same units as growth rate: inverse years. The variance itself has units of inverse years squared.

To see why covariance matters, suppose we invest a fraction $w$ in one asset and the remaining fraction $1-w$ in another, where $0<w<1$. Let $\sigma_{g,1}$ and $\sigma_{g,2}$ be their positive growth-rate standard deviations and $\rho_{12}$ their correlation. For these two assets, the variance formula becomes:
$$
\operatorname{Var}(g_p)
=w^2\sigma_{g,1}^2+(1-w)^2\sigma_{g,2}^2
+2w(1-w)\underbrace{\rho_{12}\sigma_{g,1}\sigma_{g,2}}_{\operatorname{Cov}(g_1,g_2)}.
$$
If $\rho_{12}=1$, the asset growth rates move together perfectly. The variance above is then a perfect square, so taking its square root gives:
$$
\sigma_{g,p}=w\sigma_{g,1}+(1-w)\sigma_{g,2}.
$$
With perfect correlation, the portfolio's standard deviation is just the weighted average of the two asset standard deviations. When $\rho_{12}<1$, the assets do not move together perfectly. The covariance term is smaller, and the portfolio's standard deviation falls below that weighted average. This reduction is the benefit of __diversification__.

The portfolio can even have less variance than either asset alone, provided the correlation is low enough. The cutoff is the smaller asset standard deviation divided by the larger one. For example, if the standard deviations are 2 and 4 per year, we can choose weights that reduce variance below both individual asset variances when $\rho_{12}<2/4=0.5$. The correlation can therefore be positive and still allow this reduction.

> __Growth rates and log returns:__
>
> We can also write these calculations in terms of log returns. Multiplying each growth rate by the observation interval $\Delta t$ gives the asset log returns $\mathbf{r}=\Delta t\,\mathbf{g}$. Their weighted average is $r_p=\mathbf{w}^{\top}\mathbf{r}=\Delta t\,g_p$. The expected value and variance are then given by:
> $$
> \begin{aligned}
> \mathbb{E}[r_p]&=\Delta t\,\mathbb{E}[g_p],\\
> \operatorname{Var}(r_p)&=\Delta t^2\,\operatorname{Var}(g_p).
> \end{aligned}
> $$
> Log returns are dimensionless, and their covariance is $\mathbf{\Sigma}_r=\Delta t^2\mathbf{\Sigma}_g$. Every portfolio variance is multiplied by the same positive factor $\Delta t^2$, so the portfolio with the smallest variance stays the same. If we also multiply the target growth rate by $\Delta t$, the same portfolios meet the target. We therefore obtain the same optimal weights using either growth rates or log returns.

To calculate volatility, we use the covariance rate $\mathbf{C}$ from L5b. The portfolio volatility is given by $\sqrt{\mathbf{w}^{\top}\mathbf{C}\mathbf{w}}=\sqrt{\Delta t}\,\sigma_{g,p}$ and has units of inverse square-root years. For daily observations, $\Delta t=1/252$ years, so we divide the estimated growth-rate covariance by 252: $\hat{\mathbf{C}}=\hat{\mathbf{\Sigma}}_g/252$. In this lecture, the risk axis will show the growth-rate standard deviation $\sigma_{g,p}$ in inverse years.

We now have expressions for portfolio reward and risk in terms of the weights. Let's use them to find the allocation with the smallest variance.

___


## Minimum-Variance Portfolios
We first find the allocation with the smallest variance, without requiring a particular expected growth rate. We then add a long-only constraint and a target expected growth rate to see how those requirements change the solution.

### The global minimum-variance portfolio
The __global minimum-variance (GMV) portfolio__ is the fully invested portfolio with the smallest growth-rate variance, with no requirement on its expected growth rate. We begin by allowing short positions, so the weights can be negative. The only constraint is that the weights sum to one.

For this derivation, assume the growth-rate covariance matrix $\mathbf{\Sigma}_g$ is symmetric and positive definite. Let $\mathbf{1}\in\mathbb{R}^M$ be the vector of ones, so $\mathbf{1}^{\top}\mathbf{w}=1$ expresses the budget constraint. The optimization problem is given by:
$$
\begin{aligned}
\underset{\mathbf{w}\in\mathbb{R}^M}{\operatorname{minimize}}\quad
&\frac{1}{2}\mathbf{w}^{\top}\mathbf{\Sigma}_g\mathbf{w}\\
\text{subject to}\quad &\mathbf{1}^{\top}\mathbf{w}=1.
\end{aligned}
$$
The factor of one half simplifies the derivative and does not change the minimizing weights.

> __Derivation:__
>
> We introduce a scalar Lagrange multiplier $\lambda$ for the budget constraint and form the Lagrangian:
> $$
> \mathcal{L}(\mathbf{w},\lambda)
> =\frac{1}{2}\mathbf{w}^{\top}\mathbf{\Sigma}_g\mathbf{w}
> -\lambda\left(\mathbf{1}^{\top}\mathbf{w}-1\right).
> $$
> Differentiating with respect to the weights and setting the gradient equal to zero gives:
> $$
> \nabla_{\mathbf{w}}\mathcal{L}
> =\mathbf{\Sigma}_g\mathbf{w}-\lambda\mathbf{1}
> =\mathbf{0}.
> $$
> Because $\mathbf{\Sigma}_g$ is positive definite, it has an inverse. Multiplying by that inverse gives the weights in terms of $\lambda$:
> $$
> \mathbf{w}=\lambda\mathbf{\Sigma}_g^{-1}\mathbf{1}.
> $$
> We determine $\lambda$ by requiring these weights to satisfy the budget constraint:
> $$
> \begin{aligned}
> 1&=\mathbf{1}^{\top}\mathbf{w}
> =\lambda\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{1},\\
> \lambda&=\frac{1}{\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{1}}.
> \end{aligned}
> $$
> Substituting this value into the weight expression gives:
> $$
> \boxed{
> \mathbf{w}_{\mathrm{GMV}}
> =\frac{\mathbf{\Sigma}_g^{-1}\mathbf{1}}
> {\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{1}}.
> }
> $$
> The denominator scales the weights so they sum to one. Positive definiteness also makes the variance objective strictly convex, so these weights give the unique minimum under the budget constraint.

What variance does this allocation achieve? Using $\mathbf{\Sigma}_g\mathbf{w}_{\mathrm{GMV}}=\lambda\mathbf{1}$ and $\mathbf{w}_{\mathrm{GMV}}^{\top}\mathbf{1}=1$, the minimum variance is given by:
$$
\boxed{
\begin{aligned}
\sigma_{g,\mathrm{GMV}}^2
&=\mathbf{w}_{\mathrm{GMV}}^{\top}\mathbf{\Sigma}_g\mathbf{w}_{\mathrm{GMV}}\\
&=\lambda\mathbf{w}_{\mathrm{GMV}}^{\top}\mathbf{1}
=\frac{1}{\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{1}}.
\end{aligned}
}
$$
The weights depend only on the covariance matrix because we have imposed no expected-growth requirement. When working from data, we use the estimated covariance $\hat{\mathbf{\Sigma}}_g$ in these expressions, provided it is positive definite.

Some weights may be negative, indicating short positions, as we will see in the data example. If we want a portfolio containing only long positions, we must add constraints on the weights.


### The long-only problem
For a long-only portfolio, the weights must be nonnegative and sum to one. We also require the estimated expected growth rate to be at least a target $g_{\star}$, measured in inverse years.

Using the sample-mean vector $\mathbf{g}^{\prime}$ and estimated covariance matrix $\hat{\mathbf{\Sigma}}_g$, the problem is given by:
$$
\boxed{
\begin{aligned}
\underset{\mathbf{w}\in\mathbb{R}^M}{\operatorname{minimize}}\quad
&\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_g\mathbf{w}\\
\text{subject to}\quad
&(\mathbf{g}^{\prime})^{\top}\mathbf{w}\geq g_{\star},\\
&\mathbf{1}^{\top}\mathbf{w}=1,\\
&0\leq w_i\leq1,\qquad i\in\mathcal{P}.
\end{aligned}
}
$$
This is a __convex quadratic program__: the positive semidefinite covariance matrix gives a convex quadratic objective, and the constraints are linear.

If the previous GMV weights are nonnegative and meet the growth target, they remain the solution. Otherwise, we solve the problem numerically; some weights may be zero. With the same covariance matrix, these constraints cannot lower the minimum variance because we have fewer portfolios to choose from.

In the data example, we first set $g_{\star}$ at or below the smallest asset sample mean. Every long-only portfolio meets this target because its estimated reward is a weighted average of the asset means. The solution is then the long-only GMV portfolio.

Next, we increase the target and examine how portfolio risk changes.

### The target-growth problem and the efficient frontier

How much variance must we accept to achieve a specified expected growth rate? Recall that $\boldsymbol{\mu}_g$ is the vector of asset mean growth rates. We first allow short positions and require the weights to sum to one. For each target $g_{\star}$, we find the smallest variance subject to an __equality__ growth requirement:
$$
\mathbb{E}[g_p]=\boldsymbol{\mu}_g^{\top}\mathbf{w}=g_{\star}.
$$
The minimizing portfolios form the __minimum-variance frontier__, with each target $g_{\star}$ corresponding to a different weight vector.

As in the GMV derivation, assume that $\mathbf{\Sigma}_g$ is positive definite. The asset mean growth rates must not all be equal; otherwise, changing the weights cannot change expected portfolio growth. Define four coefficients for the closed-form frontier:
$$
\begin{aligned}
a&=\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{1},\\
b&=\mathbf{1}^{\top}\mathbf{\Sigma}_g^{-1}\boldsymbol{\mu}_g,\\
c&=\boldsymbol{\mu}_g^{\top}\mathbf{\Sigma}_g^{-1}\boldsymbol{\mu}_g,\\
d&=ac-b^2.
\end{aligned}
$$
Under these assumptions, $a>0$ and $d>0$. The smallest portfolio variance at target $g_{\star}$ is given by:
$$
\boxed{
\sigma_g^2(g_{\star})
=\frac{a\,g_{\star}^2-2b\,g_{\star}+c}{d}
=\frac{1}{a}+\frac{a}{d}\left(g_{\star}-\frac{b}{a}\right)^2.
}
$$
At $g_{\star}=b/a$, the squared term vanishes and we recover the GMV variance $1/a$. Higher or lower growth targets require more variance. Taking the square root gives a hyperbola opening to the right in the risk-growth plane, with the GMV portfolio at its vertex.

__Which part of the frontier is efficient?__ A portfolio is efficient if no other feasible portfolio offers at least as much expected growth and no more variance, with a strict improvement in at least one quantity. Only the upper branch of the minimum-variance frontier, including the GMV portfolio, meets this definition.

In the figure, $p_2$ is the GMV portfolio. Portfolios $p_1$ and $p_3$ have the same standard deviation, but $p_1$ has higher expected growth. Thus, $p_3$ is dominated, as is every point on the lower branch.

The __attainable region__ contains the points reached by feasible weights. Every fully invested portfolio lies on or to the right of the hyperbola; at a fixed growth rate, moving right means more variance than the minimum.

<div>
    <center>
        <img src="figs/minimum-variance-frontier/frontier.svg" width="760" alt="Schematic of the minimum-variance frontier in the risk-growth plane: portfolio p2 is the global minimum-variance portfolio at the vertex; portfolio p1 on the solid red efficient branch and portfolio p3 on the dashed gray dominated branch have equal standard deviation, with an upward arrow showing the higher expected growth of p1"/>
    </center>
</div>

__What changes when the target is a floor?__ The package uses the sample-mean vector $\mathbf{g}^{\prime}$ to estimate expected growth and imposes the following inequality:
$$
(\mathbf{g}^{\prime})^{\top}\mathbf{w}\geq g_{\star}.
$$
Compare the target with the growth rate of the GMV portfolio under the __same weight constraints__. The long-only GMV portfolio can differ from the unconstrained one in the figure.

- At or below that growth rate, the GMV portfolio meets the target and remains the minimum-variance solution.
- Above that growth rate, a feasible target binds: the portfolio reaches the target exactly and lies on the corresponding efficient branch.

Sweeping the floor returns the GMV portfolio and efficient branch, never the dominated branch. For long-only weights, estimated growth is a weighted average of the asset sample means, so the sweep ends at the largest $g_i^{\prime}$. With the same estimated inputs, restricting weights cannot lower the minimum variance: the long-only frontier lies on or to the right of the unconstrained hyperbola, touching it where the unconstrained weights are nonnegative.

In the data example, we estimate $\mathbf{g}^{\prime}$ and $\hat{\mathbf{\Sigma}}_g$, sweep $g_{\star}$, and solve the long-only problem. We then examine how the weights change as the required growth rate increases. Let's do exactly that.

> __Example__
>
> [▶ Compute minimum-variance portfolios and the efficient frontier from data](CHEME-5660-L6a-Example-Data-MinVar-Portfolio-Fall-2026.ipynb). Estimate the inputs for a chosen set of firms from the 2014 to 2024 data, compute the global minimum-variance portfolio in closed form and with a long-only solver, sweep the target growth rate to trace the efficient frontier, compare the optimized portfolios with equal weights and an index fund on 2025 data the optimizer never saw.


___


## Estimating Portfolio Inputs

The GMV calculation uses the estimated covariance $\hat{\mathbf{\Sigma}}_g$. A target-growth calculation also uses the estimated mean growth rates $\mathbf{g}^{\prime}$. The selected weights therefore depend on the data and estimation method.

For example, the 2014 to 2024 AMD data give mean growth estimates of $0.3120$ per year from the sample mean and $0.4397$ per year from L4b's regression, a difference of $12.8$ percentage points per year. Covariance errors also affect the allocation: underestimating an asset combination's variance can make the optimizer favor it.

Our data example evaluates the resulting allocations on 2025 prices, comparing optimized portfolios with equal weights and an index fund. We can also hold the fitted model and allocation fixed and simulate possible wealth outcomes:

> __Example__
>
> [▶ Simulate a portfolio with multiple asset GBM](CHEME-5660-L6a-Example-MAGBM-Portfolio-Fall-2026.ipynb). We simulate correlated prices and the buy-and-hold wealth of a long-only minimum-variance allocation. We compare prediction bands with observed wealth and use the scaled NPV distribution to estimate the probability of exceeding a target.

This carries the trade-rule calculation from individual assets to a portfolio.

___


## Optional Advanced Material

These examples develop the frontier geometry and input-sensitivity calculations further. The [advanced index](advanced/README.md) gives a suggested order.

* [▶ Frontier geometry and the two-fund theorem](advanced/frontier-geometry/CHEME-5660-L6a-Advanced-FrontierGeometry-Fall-2026.ipynb). How can two risky frontier portfolios generate the rest of the curve? We derive the closed-form weights and show how combining two distinct frontier portfolios traces the unconstrained frontier. We then compare it with frontiers that limit short positions or require long-only weights.

* [▶ How estimates affect portfolio weights](../L6b/advanced/estimation-risk/CHEME-5660-L6b-Advanced-EstimationRisk-Fall-2026.ipynb). How much do the weights change when we estimate the inputs again? After the risk-free extension in L6b, we resample the 2014 to 2024 growth observations and compare the stability of GMV and tangent portfolios. We vary the means and covariance separately, then evaluate the resulting allocations on 2025 prices.

___


## Summary

In this lecture, we formulated the minimum-variance portfolio problem, derived the global minimum-variance weights, and traced the efficient frontier. We used a separate year of prices to evaluate the selected allocations.

> __Key Takeaways:__
>
> * **Portfolio reward and risk:** We expressed expected growth and variance in terms of the weights, mean growth rates, and covariance matrix. For two assets with positive weights, correlation below one reduced portfolio standard deviation below the weighted average of the asset standard deviations.
>
> * **Minimum-variance portfolios:** We derived the GMV weights and used target growth rates to describe the minimum-variance frontier. Its upper branch contains the efficient portfolios. We also formulated the long-only problem and explained why sweeping a growth floor returns the GMV portfolio and efficient branch.
>
> * **Estimated inputs and realized performance:** We connected the chosen weights to estimated means and covariances, then distinguished the optimization model from buy-and-hold wealth on held-out prices. An efficient training-period allocation need not have the best realized performance.

In L6b, we introduce a model for asset growth rates that separates systematic risk, associated with factors shared across assets, from firm-specific risk.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
